# ConsensusAI — Multi-Task Deep Neural Network (MT-DNN)

This notebook implements the **MT-DNN Governance Model** for ConsensusAI.
The MT-DNN acts as a predictive trust and risk evaluator inside the Memory Trust Gate.

### 7 Output Dimensions:
1. **Relevance Score** (0.0 - 1.0)
2. **Context Match Score** (0.0 - 1.0)
3. **Evidence Quality Score** (0.0 - 1.0)
4. **Temporal Validity Score** (0.0 - 1.0)
5. **Hash Integrity Score** (0.0 - 1.0)
6. **Agent Dissent Severity** (0.0 - 1.0)
7. **Deliberation Action Risk Score & Tier** (LOW, MEDIUM, HIGH, CRITICAL)

## 1. Imports & Configuration

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import json
import os
import datetime

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch Version: {torch.__version__}")

PyTorch Version: 2.5.1+cu121


## 2. Feature Extractor (MemoryTrustFeatures)

In [2]:
class FeatureExtractor:
    """Converts raw case payload, precedent memory, and agent deliberation into a 16-dim tensor."""
    def __init__(self):
        self.feature_dim = 16
        
    def extract(self, case_payload: dict, capsule_dict: dict, deliberation_dict: dict) -> torch.Tensor:
        amount = float(case_payload.get('amount', 5000.0)) / 100000.0  # Normalized amount
        risk_level = float(case_payload.get('risk_level', 0.5))
        
        age_days = float(capsule_dict.get('age_days', 2.0)) / 30.0
        base_trust = float(capsule_dict.get('base_trust_score', 0.95))
        provenance = float(capsule_dict.get('provenance_reliability', 0.90))
        evidence_cnt = float(capsule_dict.get('evidence_count', 3.0)) / 10.0
        evidence_qual = float(capsule_dict.get('evidence_quality_avg', 0.85))
        hist_success = float(capsule_dict.get('outcome_historical_success', 0.90))
        hash_verified = float(capsule_dict.get('hash_verification_result', 1.0))
        context_comp = float(capsule_dict.get('contextual_compatibility', 0.88))
        policy_diff = float(capsule_dict.get('policy_version_diff', 0.0))
        
        disagreement = float(deliberation_dict.get('agent_disagreement_index', 0.20))
        prop_var = float(deliberation_dict.get('proposal_variance', 0.15))
        conf_disp = float(deliberation_dict.get('confidence_dispersion', 0.10))
        supp_agents = float(deliberation_dict.get('supporting_agent_count', 4.0)) / 10.0
        diss_agents = float(deliberation_dict.get('dissenting_agent_count', 1.0)) / 10.0
        
        vec = [
            amount, risk_level, age_days, base_trust, provenance, evidence_cnt,
            evidence_qual, hist_success, hash_verified, context_comp, policy_diff,
            disagreement, prop_var, conf_disp, supp_agents, diss_agents
        ]
        return torch.tensor(vec, dtype=torch.float32)

extractor = FeatureExtractor()
test_vec = extractor.extract({}, {}, {})
print(f"Feature vector shape: {test_vec.shape}")

Feature vector shape: torch.Size([16])


## 3. PyTorch MT-DNN Model Architecture

In [3]:
class MTDNNModel(nn.Module):
    """Multi-Task Deep Neural Network with Shared Encoder and 7 Heads."""
    def __init__(self, input_dim: int = 16, hidden_dim: int = 64):
        super(MTDNNModel, self).__init__()
        # Shared Encoder
        self.shared_encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU()
        )
        
        # 5 Trust Heads
        self.relevance_head = nn.Sequential(nn.Linear(hidden_dim, 1), nn.Sigmoid())
        self.context_head = nn.Sequential(nn.Linear(hidden_dim, 1), nn.Sigmoid())
        self.evidence_head = nn.Sequential(nn.Linear(hidden_dim, 1), nn.Sigmoid())
        self.temporal_head = nn.Sequential(nn.Linear(hidden_dim, 1), nn.Sigmoid())
        self.hash_head = nn.Sequential(nn.Linear(hidden_dim, 1), nn.Sigmoid())
        
        # 2 Governance Heads
        self.dissent_head = nn.Sequential(nn.Linear(hidden_dim, 1), nn.Sigmoid())
        self.risk_head = nn.Sequential(nn.Linear(hidden_dim, 1), nn.Sigmoid())
        
    def forward(self, x: torch.Tensor):
        shared_repr = self.shared_encoder(x)
        
        rel = self.relevance_head(shared_repr)
        ctx = self.context_head(shared_repr)
        evd = self.evidence_head(shared_repr)
        tmp = self.temporal_head(shared_repr)
        hsh = self.hash_head(shared_repr)
        dis = self.dissent_head(shared_repr)
        rsk = self.risk_head(shared_repr)
        
        return torch.cat([rel, ctx, evd, tmp, hsh, dis, rsk], dim=-1)

model = MTDNNModel()
out = model(test_vec.unsqueeze(0))
print(f"MT-DNN Forward Output Shape: {out.shape}")
print(f"Initial predictions: {out.squeeze().detach().tolist()}")

MT-DNN Forward Output Shape: torch.Size([1, 7])
Initial predictions: [0.5531103610992432, 0.41770052909851074, 0.42749670147895813, 0.2641771137714386, 0.6403869986534119, 0.7290279269218445, 0.4272310435771942]


## 4. Synthetic Training Dataset Generation

In [4]:
def generate_synthetic_dataset(num_samples: int = 500):
    X = []
    Y = []
    for _ in range(num_samples):
        hash_val = 1.0 if np.random.rand() > 0.1 else 0.0
        age = np.random.uniform(0, 1.0)
        base_t = np.random.uniform(0.5, 1.0)
        evd_q = np.random.uniform(0.4, 1.0)
        diss = np.random.uniform(0.0, 0.9)
        amt = np.random.uniform(0.1, 1.0)
        
        vec = [
            amt, np.random.rand(), age, base_t, np.random.rand(), evd_q,
            evd_q, base_t, hash_val, np.random.rand(), 0.0,
            diss, diss * 0.8, diss * 0.5, 0.8 - diss*0.5, diss*0.5
        ]
        
        # Targets: [rel, ctx, evd, tmp, hsh, dis, rsk]
        rel_target = base_t * 0.9
        ctx_target = 1.0 - abs(amt - 0.5)
        evd_target = evd_q
        tmp_target = max(0.1, 1.0 - age * 0.8)
        hsh_target = hash_val
        dis_target = diss
        rsk_target = min(1.0, diss * 0.5 + (1.0 - hash_val) * 0.9 + amt * 0.3)
        
        X.append(vec)
        Y.append([rel_target, ctx_target, evd_target, tmp_target, hsh_target, dis_target, rsk_target])
        
    return torch.tensor(X, dtype=torch.float32), torch.tensor(Y, dtype=torch.float32)

X_train, Y_train = generate_synthetic_dataset(500)
dataset = TensorDataset(X_train, Y_train)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
print(f"Generated dataset: X={X_train.shape}, Y={Y_train.shape}")

Generated dataset: X=torch.Size([500, 16]), Y=torch.Size([500, 7])


## 5. Multi-Task Training Loop

In [5]:
loss_weights = torch.tensor([1.0, 1.0, 1.0, 1.0, 2.0, 1.2, 1.5])
criterion = nn.MSELoss(reduction='none')
optimizer = optim.Adam(model.parameters(), lr=0.005)

num_epochs = 15
print("Starting MT-DNN Training...")
for epoch in range(num_epochs):
    total_loss = 0.0
    for batch_x, batch_y in dataloader:
        optimizer.zero_grad()
        preds = model(batch_x)
        raw_loss = criterion(preds, batch_y)
        weighted_loss = (raw_loss * loss_weights).mean()
        weighted_loss.backward()
        optimizer.step()
        total_loss += weighted_loss.item()
        
    if (epoch + 1) % 3 == 0 or epoch == num_epochs - 1:
        print(f"Epoch [{epoch+1}/{num_epochs}] - Total Weighted Loss: {total_loss/len(dataloader):.6f}")

# Save checkpoint
checkpoint_path = os.path.join(os.path.dirname(__file__), "mtdnn_checkpoint.pt") if '__file__' in locals() else "mtdnn_checkpoint.pt"
torch.save({
    'model_state_dict': model.state_dict(),
    'version': 'v1.0.0',
    'timestamp': datetime.datetime.now(datetime.UTC).isoformat()
}, checkpoint_path)
print(f"Saved MT-DNN model checkpoint to {checkpoint_path}")

Starting MT-DNN Training...
Epoch [3/15] - Total Weighted Loss: 0.010880
Epoch [6/15] - Total Weighted Loss: 0.005019
Epoch [9/15] - Total Weighted Loss: 0.003924
Epoch [12/15] - Total Weighted Loss: 0.002947
Epoch [15/15] - Total Weighted Loss: 0.002329
Saved MT-DNN model checkpoint to mtdnn_checkpoint.pt


## 6. Risk Tiering & Inference Demo

In [6]:
def predict_trust_and_risk(model, features_tensor: torch.Tensor):
    model.eval()
    with torch.no_grad():
        if features_tensor.ndim == 1:
            features_tensor = features_tensor.unsqueeze(0)
        out = model(features_tensor).squeeze().tolist()
        
        rel, ctx, evd, tmp, hsh, dis, rsk = out
        
        # Risk tiering
        if rsk < 0.35:
            tier = "LOW"
        elif rsk < 0.65:
            tier = "MEDIUM"
        elif rsk < 0.85:
            tier = "HIGH"
        else:
            tier = "CRITICAL"
            
        return {
            "relevance_score": round(rel, 4),
            "context_match_score": round(ctx, 4),
            "evidence_quality_score": round(evd, 4),
            "temporal_validity_score": round(tmp, 4),
            "hash_integrity_score": round(hsh, 4),
            "dissent_severity": round(dis, 4),
            "action_risk_score": round(rsk, 4),
            "action_risk_tier": tier,
            "model_version": "v1.0.0"
        }

sample_features = extractor.extract({'amount': 75000}, {'base_trust_score': 0.92}, {'agent_disagreement_index': 0.65})
res = predict_trust_and_risk(model, sample_features)
print("Sample MT-DNN Inference Output:")
print(json.dumps(res, indent=2))

Sample MT-DNN Inference Output:
{
  "relevance_score": 0.778,
  "context_match_score": 0.6948,
  "evidence_quality_score": 0.6391,
  "temporal_validity_score": 0.8562,
  "hash_integrity_score": 0.9971,
  "dissent_severity": 0.4827,
  "action_risk_score": 0.4591,
  "action_risk_tier": "MEDIUM",
  "model_version": "v1.0.0"
}
